# 2.1 理论计算题
## 题目已知条件
1. 字符序列：`ababc`
2. 词汇表 $V=\{a,b,c\}$，词汇大小 $|V|=3$
3. 一阶马尔可夫模型，拉普拉斯（加1）平滑转移概率公式：
$$
p(y|x) = \frac{count(x,y)+1}{Total(x)+|V|}
$$
- $count(x,y)$：转移对 $(x,y)$ 出现频次
- $Total(x)$：以字符 $x$ 作为前驱的全部转移总次数

## 步骤1：提取相邻转移二元组
序列拆分：a b a b c
相邻转移对共4组：$(a,b),(b,a),(a,b),(b,c)$

## 步骤2：统计转移计数矩阵
| 前驱字符 x \ 后继字符 y | a | b | c |
| :---------------------: |:-:|:-:|:-:|
| a                       | 0 | 2 | 0 |
| b                       | 1 | 0 | 1 |
| c                       | 0 | 0 | 0 |

## 步骤3：计算各前驱总转移次数
- $Total(a) = count(a,a)+count(a,b)+count(a,c) = 0+2+0 = 2$
- $Total(b) = count(b,a)+count(b,b)+count(b,c) = 1+0+1 = 2$
- $Total(c) = count(c,a)+count(c,b)+count(c,c) = 0+0+0 = 0$

## 步骤4：求解条件概率
### 1. $p(\text{'a'} \mid \text{'b'})$
$$
p(a|b) = \frac{count(b,a)+1}{Total(b)+|V|}
= \frac{1+1}{2+3}
= \frac{2}{5} = 0.4
$$

### 2. $p(\text{'c'} \mid \text{'b'})$
$$
p(c|b) = \frac{count(b,c)+1}{Total(b)+|V|}
= \frac{1+1}{2+3}
= \frac{2}{5} = 0.4
$$

## 校验（概率归一性）
以 `b` 为前驱的全部转移概率：
$$
p(a|b)=\frac{2}{5},\quad p(b|b)=\frac{0+1}{2+3}=\frac{1}{5},\quad p(c|b)=\frac{2}{5}
$$
求和：$\displaystyle \frac{2}{5}+\frac{1}{5}+\frac{2}{5}=1$，满足概率归一要求。

## 最终答案
1. $p(\text{'a'} \mid \text{'b'}) = \boldsymbol{\dfrac{2}{5}}$
2. $p(\text{'c'} \mid \text{'b'}) = \boldsymbol{\dfrac{2}{5}}$

# 2.2 编程题

In [11]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 转小写并去除标点，仅保留字母和空格
    lower_text = text.lower()
    clean_text = ''.join([c for c in lower_text if c.isalpha() or c == ' '])
    # 分词
    words = clean_text.split()
    # 构建词频并生成词汇表
    cnt = Counter(words)
    sorted_words = sorted(cnt.keys(), key=lambda x: (-cnt[x], words.index(x)))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    # 滑动窗口构造特征与标签
    features = []
    labels = []
    total = len(words)
    for i in range(total - n + 1):
        window = words[i:i+n]
        features.append(window)
        next_pos = i + n
        label = words[next_pos] if next_pos < total else None
        labels.append(label)
    return vocab, (features, labels)

# 测试
if __name__ == "__main__":
    vocab, (feat, lab) = preprocess_text("The time machine", 2)
    print(vocab)
    print(feat)
    print(lab)

{'the': 0, 'time': 1, 'machine': 2}
[['the', 'time'], ['time', 'machine']]
['machine', None]


# 3.1 理论计算题
## 题目已知条件
1. 无偏置线性RNN：$h_t = W_{hh}h_{t-1} + W_{hx}x_t$
2. 输出层映射：$o_t = W_{oh}h_t$
3. 平方损失函数：$\displaystyle L = \frac12 \sum_{t=1}^T (o_t - y_t)^2$
4. 使用BPTT沿时间反向传播，推导损失对$W_{hh}$完整梯度，说明梯度消失/爆炸条件
- $l_t=\frac12(o_t-y_t)^2$：单时间步损失，总损失 $L=\sum_{t=1}^T l_t$
- $\delta_t=\frac{\partial l_t}{\partial h_t}$：单步损失对当前隐状态梯度

## 步骤1：计算局部基础梯度
$$
\frac{\partial l_t}{\partial o_t} = o_t - y_t
$$
$$
\delta_t = \frac{\partial l_t}{\partial h_t} = \frac{\partial l_t}{\partial o_t} \cdot \frac{\partial o_t}{\partial h_t} = (o_t-y_t)W_{oh}^\top
$$

## 步骤2：隐状态时序梯度递推关系
由递推式 $h_{k+1}=W_{hh}h_k + W_{hx}x_{k+1}$，可得隐状态间导数：
$$
\frac{\partial h_{k+1}}{\partial h_k} = W_{hh}
$$
对$t$时刻损失反向传递至更早时刻$k<t$：
$$
\frac{\partial l_t}{\partial h_k} = \delta_t \cdot (W_{hh})^{t-k}
$$
累加全部时间步得到总梯度：
$$
\frac{\partial L}{\partial h_k} = \delta_k + \delta_{k+1}W_{hh} + \delta_{k+2}(W_{hh})^2 + \dots + \delta_T(W_{hh})^{T-k}
$$

## 步骤3：推导损失对$W_{hh}$完整梯度
任意时刻$t$，$h_t$关于$W_{hh}$的导数：
$$
\frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top
$$
遍历所有时间步、所有前驱时刻累加得到完整梯度：
$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^{t-1} \big(W_{hh}^\top\big)^{t-k-1} \delta_t \cdot h_k^\top
$$

## 步骤4：梯度消失与爆炸判定条件
设$\rho(W_{hh})$为$W_{hh}$矩阵的谱半径（所有特征值模的最大值）：
1. 梯度爆炸：$\rho(W_{hh})>1$，$(W_{hh})^n$随时间步指数放大，梯度数值无限增大；
2. 梯度消失：$\rho(W_{hh})<1$，$(W_{hh})^n$随时间步指数趋近0，早期时序信息梯度趋近于0；
3. 临界稳定：$\rho(W_{hh})=1$，梯度无指数缩放，训练中几乎不会出现。

## 最终结论
损失对权重$W_{hh}$梯度表达式：
$$
\boldsymbol{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \sum_{k=1}^{t-1} \big(W_{hh}^\top\big)^{t-k-1} \delta_t \, h_k^\top}
$$
梯度爆炸条件：$\rho(W_{hh})>1$；梯度消失条件：$\rho(W_{hh})<1$

# 3.2 编程题

In [18]:
import numpy as np

class SimpleRNNCell:
    def forward(self, x_t, h_prev, W_hx, W_hh, b_h):
        """
        前向传播
        :param x_t: (batch_size, input_size)
        :param h_prev: (batch_size, hidden_size)
        :param W_hx: (hidden_size, input_size)
        :param W_hh: (hidden_size, hidden_size)
        :param b_h: (1, hidden_size)
        :return h_t: 当前隐状态, cache: 反向传播所需缓存
        """
        # 计算预激活值
        pre_activation = x_t @ W_hx.T + h_prev @ W_hh.T + b_h
        # tanh激活
        h_t = np.tanh(pre_activation)
        # 保存中间变量用于反向传播
        cache = (x_t, h_prev, W_hx, W_hh, b_h, pre_activation)
        return h_t, cache

    def backward(self, dh_next, cache):
        """
        单步反向传播，计算各梯度
        :param dh_next: dL/dh_t, shape (batch_size, hidden_size)
        :param cache: 前向传播缓存
        :return dx_t, dh_prev, dW_hx, dW_hh, db_h
        """
        x_t, h_prev, W_hx, W_hh, b_h, pre_activation = cache
        batch_size = x_t.shape[0]

        # tanh导数: d(tanh(z))/dz = 1 - tanh(z)^2
        tanh_deriv = 1 - np.tanh(pre_activation) ** 2
        d_pre = dh_next * tanh_deriv  # dL/d(pre_activation)

        # 输入梯度 dx_t
        dx_t = d_pre @ W_hx
        # 上一时刻隐状态梯度 dh_prev
        dh_prev = d_pre @ W_hh
        # 权重梯度
        dW_hx = d_pre.T @ x_t
        dW_hh = d_pre.T @ h_prev
        # 偏置梯度，batch维度求和
        db_h = np.sum(d_pre, axis=0, keepdims=True)

        return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 测试示例
if __name__ == "__main__":
    # 超参
    batch = 4
    input_dim = 3
    hidden_dim = 5

    # 随机初始化输入与权重
    x = np.random.randn(batch, input_dim)
    h_prev = np.random.randn(batch, hidden_dim)
    Whx = np.random.randn(hidden_dim, input_dim)
    Whh = np.random.randn(hidden_dim, hidden_dim)
    bh = np.random.randn(1, hidden_dim)

    cell = SimpleRNNCell()
    # 前向
    ht, cache = cell.forward(x, h_prev, Whx, Whh, bh)
    # 模拟上游梯度
    dht = np.random.randn(batch, hidden_dim)
    # 反向求梯度
    dx, dhp, dWhx, dWhh, dbh = cell.backward(dht, cache)

    print("h_t shape:", ht.shape)
    print("dx_t shape:", dx.shape)
    print("dh_prev shape:", dhp.shape)
    print("dW_hx shape:", dWhx.shape)
    print("dW_hh shape:", dWhh.shape)
    print("db_h shape:", dbh.shape)

h_t shape: (4, 5)
dx_t shape: (4, 3)
dh_prev shape: (4, 5)
dW_hx shape: (5, 3)
dW_hh shape: (5, 5)
db_h shape: (1, 5)


# 4.1 理论计算题
## 题目已知条件
1. 深度双向RNN：层数$L$，单层隐藏单元数$H$，输入维度$D$，输出维度$O$
2. 忽略嵌入层、输出前置投影层，仅统计全部循环层与最终输出层权重、偏置
3. 每一层包含独立前向RNN、后向RNN两套参数

## 步骤1：单层双向RNN参数统计
前向RNN参数：$W_{hx}^f(H\times D),\ W_{hh}^f(H\times H),\ b_h^f(1\times H)$
后向RNN参数：$W_{hx}^b(H\times D),\ W_{hh}^b(H\times H),\ b_h^b(1\times H)$
单层双向RNN总参数：
$$
2HD + 2H^2 + 2H
$$

## 步骤2：$L$层循环层总参数
$$
L \cdot \big(2HD + 2H^2 + 2H\big)
$$

## 步骤3：顶层输出层参数
每层输出拼接前向、后向隐状态，维度为$2H$；输出权重$W_o(2H\times O)$，偏置$b_o(1\times O)$
$$
2HO + O
$$

## 步骤4：模型全部参数汇总表达式
$$
\text{TotalParams} = L\big(2HD + 2H^2 + 2H\big) + 2HO + O
$$

## 最终答案
模型总参数完整表达式：
$$
\boldsymbol{\text{TotalParams} = L(2HD + 2H^2 + 2H) + O(2H + 1)}
$$


# 4.2 编程题

In [19]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 双向RNN，batch_first=False 匹配输入形状(seq_len, batch, input_dim)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            batch_first=False
        )

    def forward(self, X):
        """
        :param X: 输入序列 (seq_len, batch, input_dim)
        :return:
            out: 每一时间步拼接正反隐状态 (seq_len, batch, 2 * hidden_dim)
            final_h: 最后一步拼接隐状态 (batch, 2 * hidden_dim)
        """
        # out: (seq_len, batch, 2*hidden_dim)
        # hn: (num_directions, batch, hidden_dim)
        out, hn = self.rnn(X)

        # 拼接前向最后一步、后向最后一步
        forward_h = hn[0]
        backward_h = hn[1]
        final_h = torch.cat([forward_h, backward_h], dim=-1)

        return out, final_h


# 测试代码
if __name__ == "__main__":
    seq_len, batch, input_dim = 10, 3, 4
    hidden_dim = 6
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch, input_dim)
    seq_output, last_h = encoder(X)
    print("每步拼接隐状态 shape:", seq_output.shape)  # (10, 3, 12)
    print("最终拼接隐状态 shape:", last_h.shape)       # (3, 12)

每步拼接隐状态 shape: torch.Size([10, 3, 12])
最终拼接隐状态 shape: torch.Size([3, 12])


# 5.1 理论计算题
## 题目已知条件
1. Skip-gram模型，中心词向量$v_c$，真实上下文词向量$u_o$
2. 负采样机制：每次采样$K$个负样本，负样本词向量$u_{n_k}$
3. sigmoid激活：$\displaystyle \sigma(z)=\frac{1}{1+e^{-z}}$，目标为负对数似然损失

## 步骤1：单组中心词-上下文样本损失
最大化真实词对相似度，压低负样本相似度，单个样本损失：
$$
\mathcal{L}(w_c,w_o) = -\left[\log\sigma(v_c^\top u_o) + \sum_{k=1}^K \log\big(1-\sigma(v_c^\top u_{n_k})\big)\right]
$$

## 步骤2：全局完整目标函数
遍历语料中全部中心词-上下文词样本对集合$\mathcal{D}$，总损失：
$$
\mathcal{J} = -\sum_{(w_c,w_o)\in \mathcal{D}} \left[ \log\sigma(v_c^\top u_o) + \sum_{k=1}^K \log\big(1-\sigma(v_c^\top u_{n_k})\big) \right]
$$

## 步骤3：负样本采样规则
采用词频平滑噪声分布进行采样：
$$
P_n(w) \propto U(w)^{\frac{3}{4}}
$$
$U(w)$代表单词在语料中的原始出现频次，训练时从该分布抽取$K$个不等于当前上下文$w_o$的单词作为负样本。

## 最终答案
完整全局目标函数：
$$
\boldsymbol{\mathcal{J} = -\sum_{(w_c,w_o)\in \mathcal{D}} \left[ \log\sigma(v_c^\top u_o) + \sum_{k=1}^K \log\big(1-\sigma(v_c^\top u_{n_k})\big) \right]}
$$
负样本从分布$P_n(w) \propto U(w)^{\frac{3}{4}}$中随机采样。

# 5.2 编程题

In [20]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_ids, target_idx, W, W_out):
    """
    CBOW 前向传播 + 完整softmax交叉熵损失
    :param context_ids: list[int]，单个样本上下文词索引列表，长度context_size
    :param target_idx: int，中心词索引
    :param W: 嵌入矩阵 (V, d)
    :param W_out: 输出权重 (d, V)
    :return: loss 标量损失值
    """
    # 1. 取出所有上下文词嵌入
    ctx_embeds = W[torch.tensor(context_ids)]  # (context_size, d)
    # 2. 上下文向量求平均得到隐藏层
    hidden = torch.mean(ctx_embeds, dim=0, keepdim=True)  # (1, d)
    # 3. 计算输出logits
    logits = hidden @ W_out  # (1, V)
    # 4. 完整softmax交叉熵损失
    loss = F.cross_entropy(logits, torch.tensor([target_idx]))
    return loss

# 测试示例
if __name__ == "__main__":
    V = 10    # 词汇量
    d = 4     # 嵌入维度
    context_size = 3
    W = torch.randn(V, d)
    W_out = torch.randn(d, V)

    ctx = [0, 2, 5]
    target = 3
    loss_val = cbow_forward_loss(ctx, target, W, W_out)
    print("CBOW loss:", loss_val.item())

CBOW loss: 1.014258861541748


# 6.1 理论计算题
## 题目已知条件
1. 矩阵维度：$Q \in \mathbb{R}^{2\times4},\ K \in \mathbb{R}^{3\times4},\ V \in \mathbb{R}^{3\times5}$
2. 键维度$d_k=4$，缩放因子$\sqrt{d_k}=2$
3. 无掩码缩放点积注意力公式：
$$
\text{Score} = \frac{QK^\top}{\sqrt{d_k}},\quad \text{Attention}(Q,K,V) = \text{softmax}(\text{Score})V
$$

## 步骤1：计算原始点积得分矩阵$QK^\top$
$Q$形状$2\times4$，$K^\top$形状$4\times3$，矩阵相乘结果$S_{\text{raw}}=QK^\top \in \mathbb{R}^{2\times3}$
$$
S_{\text{raw}} =
\begin{bmatrix}
q_1 k_1^\top & q_1 k_2^\top & q_1 k_3^\top \\
q_2 k_1^\top & q_2 k_2^\top & q_2 k_3^\top
\end{bmatrix}
$$
其中$q_i k_j^\top = \sum_{m=1}^4 q_{i,m}k_{j,m}$，代表第$i$个查询与第$j$个键的内积。

## 步骤2：缩放得到标准化得分矩阵
$$
\text{Score} = \frac{S_{\text{raw}}}{\sqrt{d_k}} = \frac{S_{\text{raw}}}{2},\quad \text{Score} \in \mathbb{R}^{2\times3}
$$

## 步骤3：按行执行Softmax归一，得到注意力权重矩阵
对得分矩阵每一行独立做Softmax，每行元素之和为1，权重矩阵$A\in\mathbb{R}^{2\times3}$：
$$
A_{i,j} = \frac{\exp\big(\text{Score}_{i,j}\big)}{\sum_{m=1}^3 \exp\big(\text{Score}_{i,m}\big)}
$$

## 步骤4：加权求和得到注意力输出矩阵
权重矩阵$A(2\times3)$与值矩阵$V(3\times5)$相乘，输出形状$\mathbb{R}^{2\times5}$：
$$
\text{Output} = A \cdot V
$$

## 最终答案
输出矩阵尺寸$\boldsymbol{\mathbb{R}^{2\times5}}$，完整计算链路：
$$
\boldsymbol{\text{Output} = \text{softmax}\left(\frac{QK^\top}{\sqrt{4}}\right)V}

# 6.2 编程题

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.d_model = 4
        self.num_heads = 2
        self.d_k = self.d_model // self.num_heads  # d_k = 2
        # QKV 投影层
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        # 输出融合线性层
        self.w_o = nn.Linear(self.d_model, self.d_model)

    def scaled_dot_product_attn(self, q, k, v):
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weight = F.softmax(attn_score, dim=-1)
        return torch.matmul(attn_weight, v)

    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape

        # 线性投影
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)

        # 分头: (seq_len, batch, heads, d_k) -> (seq_len, heads, batch, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).transpose(1, 2)

        # 单头注意力计算
        attn_out = self.scaled_dot_product_attn(Q, K, V)  # (seq_len, heads, batch, d_k)

        # 拼接多头
        attn_out = attn_out.transpose(1, 2).contiguous()  # (seq_len, batch, heads, d_k)
        concat_out = attn_out.view(seq_len, batch, self.d_model)

        # 最终线性变换
        output = self.w_o(concat_out)
        return output

# 测试代码
if __name__ == "__main__":
    mha = MultiHeadAttention()
    seq_len, batch = 5, 3
    X = torch.randn(seq_len, batch, 4)
    out = mha(X)
    print("输出shape:", out.shape)  # torch.Size([5, 3, 4])

输出shape: torch.Size([5, 3, 4])
